# Chapter 20: Merge Queues (Reference)

> ⭐ **Optional section** — feel free to skip on first pass.

## Learning Objectives

- Simulate naive parallel auto-merge and observe a semantic conflict slip through
- Simulate a merge queue and watch it catch the same conflict before merging
- Explain why each PR's own CI run cannot catch a semantic conflict
- State the concrete signal that would mean this repo needs a queue

## Setup

The next cell sets up reproducibility and the `PRA_MODE` toggle. You should see `PRA_MODE = 'fixture'` printed by default.

In [1]:
import os
import random
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pr_automerge").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

RANDOM_STATE: int = 42
random.seed(RANDOM_STATE)

PRA_MODE = os.environ.get("PRA_MODE", "fixture")
PRA_REPO = os.environ.get("PRA_REPO", "")

if PRA_MODE == "live":
    assert PRA_REPO, "Set PRA_REPO=owner/name to run against a real repo"

print(f"PRA_MODE = {PRA_MODE!r}")

PRA_MODE = 'fixture'


## Advanced: Simulating the Two Strategies

> ⭐ **Optional section** — feel free to skip on first pass.

The next cell builds three sample PRs, two of which share a conceptual resource (`config/feature_flags.yml`), and runs both `simulate_parallel_automerge` and `simulate_merge_queue` against them. You should see the parallel strategy merge all three and only THEN report a conflict, while the queue holds PR #202 before it ever merges.

In [2]:
from labs.lab_20_merge_queues import (
    QueuedPR,
    simulate_parallel_automerge,
    simulate_merge_queue,
)

prs = [
    QueuedPR(201, resources_touched={"config/feature_flags.yml"}, passes_ci_alone=True),
    QueuedPR(202, resources_touched={"config/feature_flags.yml"}, passes_ci_alone=True),
    QueuedPR(203, resources_touched={"sandbox/app/greeting.py"}, passes_ci_alone=True),
]

parallel_result = simulate_parallel_automerge(prs)
print(f"parallel merged: {parallel_result['merged']}")
print(
    f"conflict detected AFTER merge: {parallel_result['conflict_detected_after_merge']}"
)

queue_result = simulate_merge_queue(prs)
for row in queue_result:
    status = "MERGED" if row["merged"] else f"HELD ({row['reason']})"
    print(f"queue PR #{row['number']}: {status}")

parallel merged: [201, 202, 203]
conflict detected AFTER merge: True
queue PR #201: MERGED
queue PR #202: HELD (conflicts with queue: {'config/feature_flags.yml'})
queue PR #203: MERGED


## Takeaways & Next Steps

This notebook's takeaway is the timing difference above -- the parallel strategy discovers the conflict only after both PRs are already merged; the queue never lets the second one land.

In [3]:
print(
    "This repo doesn't need a queue -- see Chapter 20 Section 12 for the concrete signal that would change that."
)

This repo doesn't need a queue -- see Chapter 20 Section 12 for the concrete signal that would change that.


---

📖 **Reading companion:** [Chapter 20: Merge Queues](../learning_modules/chapter_20_merge_queues.md)
